# 팀 공유용 데이터 내보내기 (export_clean_dataset)

`run_experiment_augmented.ipynb`의 1~4단계(leakage-safe 파이프라인)를 그대로
재현해 `train_aug`/`val_aug`를 만든 뒤, **심볼릭 링크가 아니라 실제 이미지
파일을 복사**해서 팀원에게 통째로 zip으로 넘길 수 있는 자기완결적 데이터셋으로
내보낸다.

왜 필요한가: 지금까지 쓰던 `prepare_yolo_dataset(..., symlink=True)`는 이미지에
심볼릭 링크만 걸어두는데, 그 링크가 가리키는 원본은 이 컴퓨터의 Kaggle/AIHub
다운로드 경로(예: `~/.cache/kagglehub/...`, 개인 다운로드 폴더)라서 다른 팀원
컴퓨터에서는 열리지 않는다. **이 노트북은 그 심볼릭 링크 워크플로우를 건드리지
않는다** — 완전히 별개로, 공유용 사본만 새로 만든다.

전체 데이터 누수 제거 과정에 대한 설명은 `docs/data_leakage_prevention.md` 참고.
이 노트북은 그 문서의 5단계 파이프라인을 그대로 실행할 뿐이다(재현성을 위해
seed=42 등 동일 설정).

In [ ]:
from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root()
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
from pathlib import Path

from src.audit_leakage import audit
from src.data.aihub_converter import convert_aihub_to_coco
from src.data.kaggle_converter import convert_kaggle_to_coco
from src.data.merge import merge_for_augmentation
from src.data.subset import (
    build_leakage_safe_groups,
    create_subset,
    exclude_images,
    export_portable_dataset,
)

CONFIGS_DIR = PROJECT_ROOT / "configs" / "experiment"
DATA_DIR = PROJECT_ROOT / "data"
EXPORT_DIR = DATA_DIR / "export_shared_clean"  # 이 노트북의 결과물(팀 공유용)
AIHUB_COMBOS = [1, 3, 4, 5, 6]

print("EXPORT_DIR:", EXPORT_DIR)


## 1단계 — Kaggle train → COCO, AIHub 조합 변환

`run_experiment_augmented.ipynb`와 동일.

In [ ]:
kaggle_coco = convert_kaggle_to_coco()
allowed_ids = {c["id"] for c in kaggle_coco["categories"]}  # 제출 대상 56
print("Kaggle 클래스:", len(allowed_ids))

aihub_coco_raw = convert_aihub_to_coco(combo_nums=AIHUB_COMBOS)


## 2단계 — 실제 Kaggle 테스트셋 대비 leakage 감사 + AIHub 필터링

학습에 넣으려는 AIHub 이미지가 실제 Kaggle 비공개 테스트셋과 근접중복인지
확인하고, 근접중복이면 아예 내보내기 대상에서도 제외한다(진짜 컨닝 위험).

In [ ]:
audit_result = audit(combos=AIHUB_COMBOS)
print("\n판정:", audit_result["verdict"])
print(
    "AIHub 근접중복(실제 테스트셋과):",
    len(audit_result["aihub_leak_paths"]),
    "장 → 제외 예정",
)

aihub_coco = exclude_images(aihub_coco_raw, audit_result["aihub_leak_paths"])


## 3단계 — 근접중복 인식 그룹 구성 (누수 안전 그룹)

`combo_group_key`만 쓰면 조합 문자열이 다른 근접중복 장면(같은 트레이에서 약
1개만 교체)이 train/val로 갈릴 수 있다. `build_leakage_safe_groups`로 Kaggle
전체 + (필터링된) AIHub 이미지 전체를 대상으로 지각 해시 기반 그룹을 미리 만들어
아래 4단계에 넘긴다.

In [ ]:
_pool_images = kaggle_coco["images"] + aihub_coco["images"]
group_overrides = build_leakage_safe_groups({
    "images": _pool_images,
    "annotations": [],
    "categories": [],
})


## 4단계 — train/val 분할 + AIHub 증강 병합 (그룹화 fix 적용)

val은 Kaggle 대표 검증셋으로 유지하되, `group_overrides`를 넘겨 근접중복 장면이
train/val로 갈리지 않도록 한다. `run_experiment_augmented.ipynb`와 완전히 동일한
seed(42)를 쓰므로, 이렇게 만든 `train_aug`/`val_aug`는 현재 실험에 쓰이는
데이터와 정확히 같다.

In [ ]:
train_k, val_k = create_subset(
    kaggle_coco, tier="full", test_size=0.2, seed=42, group_overrides=group_overrides
)

train_aug, val_aug = merge_for_augmentation(
    base_train=train_k,
    base_val=val_k,
    extra_coco=aihub_coco,
    allowed_ids=allowed_ids,
    drop_pure_irrelevant=True,
    group_overrides=group_overrides,
)


### 검증 — 근접중복 재점검

내보내기 전에 train/val 사이 근접중복이 남아있지 않은지 다시 확인한다. 0%에
가까워야 한다(0이 아니면 내보내기를 멈추고 원인을 먼저 확인할 것).

In [ ]:
from src.audit_leakage import _dhash, _gray_vec, _rmse


def audit_near_duplicates(
    train_coco, val_coco, hash_size=16, hash_cutoff=12, rmse_cutoff=0.06
):
    """train/val 사이 지각 해시 근접 중복(사실상 같은 장면일 가능성)을 스캔."""
    train_paths = [Path(im["file_name"]) for im in train_coco["images"]]
    val_paths = [Path(im["file_name"]) for im in val_coco["images"]]

    train_dh = [(_dhash(p, size=hash_size), p) for p in train_paths]
    train_dh = [(d, p) for d, p in train_dh if d is not None]

    near_dups, affected_val = [], set()
    for vp in val_paths:
        vd = _dhash(vp, size=hash_size)
        if vd is None or not train_dh:
            continue
        best_d, best_p = min(
            ((bin(vd ^ d).count("1"), p) for d, p in train_dh), key=lambda x: x[0]
        )
        if best_d <= hash_cutoff:
            vv, sv = _gray_vec(vp), _gray_vec(best_p)
            if vv is not None and sv is not None and _rmse(vv, sv) <= rmse_cutoff:
                near_dups.append((vp, best_p, best_d, _rmse(vv, sv)))
                affected_val.add(vp)

    pct = 100 * len(affected_val) / max(len(val_paths), 1)
    print(
        f"val {len(val_paths)}장 중 근접중복 의심 {len(affected_val)}장({pct:.1f}%) | "
        f"총 매칭 {len(near_dups)}쌍"
    )
    return near_dups


_near_dups = audit_near_duplicates(train_aug, val_aug)
assert len(_near_dups) == 0, (
    "train/val 근접중복이 남아있음 — 내보내기 전에 원인을 먼저 확인할 것"
)
print("근접중복 0건 확인 — 내보내기 진행 가능")


## 5단계 — 실제 파일로 내보내기 (심볼릭 링크 아님)

`export_portable_dataset()`이 이미지 바이트를 실제로 복사하고, COCO json의
`file_name`을 `EXPORT_DIR` 기준 **상대경로**로 다시 써서 저장한다. 이 폴더를
통째로 zip해서 넘기면, 받은 사람은 어디에 압축을 풀든 `load_portable_coco()`로
바로 쓸 수 있다(기존 심볼릭 링크 기반 `data/yolo_aug_clean/` 등은 그대로 둔다 —
이 노트북은 별개의 새 폴더에만 씀).

In [ ]:
train_json, val_json = export_portable_dataset(train_aug, val_aug, EXPORT_DIR)


### 매니페스트 저장 (팀원이 무엇을 받는지 알 수 있도록)

In [ ]:
import json as _json
from datetime import datetime

manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "pipeline": "run_experiment_augmented.ipynb 1~4단계와 동일(leakage-safe)",
    "aihub_combos": AIHUB_COMBOS,
    "seed": 42,
    "train_images": len(train_aug["images"]),
    "val_images": len(val_aug["images"]),
    "num_classes": len(train_aug["categories"]),
    "aihub_test_set_leaks_excluded": len(audit_result["aihub_leak_paths"]),
    "train_val_near_dup_pairs_after_split": len(_near_dups),
}
with open(EXPORT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    _json.dump(manifest, f, ensure_ascii=False, indent=2)

readme_text = f"""# 알약 검출 학습 데이터 (leakage-safe export)

Kaggle train + AIHub 조합 {AIHUB_COMBOS} 중, 실제 Kaggle 테스트셋과 근접중복인
이미지를 제외하고, train/val 사이 근접중복도 없도록 그룹 분할한 결과물이다.
생성 과정 전체 설명은 CJY의 `docs/data_leakage_prevention.md` 참고.

- train: {len(train_aug["images"])}장, val: {len(val_aug["images"])}장, 클래스 {len(train_aug["categories"])}개
- 생성일: {manifest["created_at"]}
- 상세 수치: manifest.json 참고

## 사용법

```python
from src.data.subset import load_portable_coco

train_coco = load_portable_coco("train.json")
val_coco = load_portable_coco("val.json")
```

`load_portable_coco`가 file_name을 이 폴더 기준 절대경로로 되돌려주므로, 폴더를
어디에 두든(압축 해제 위치 무관) 그대로 쓸 수 있다.

- torchvision 계열: `src.data.dataset.PillDetectionDataset(train_coco, ...)`에 바로 전달.
- YOLO가 필요하면 로컬에서 한 번 아래처럼 변환:
  ```python
  from src.data.coco_to_yolo import prepare_yolo_dataset
  prepare_yolo_dataset(train_coco, val_coco, "yolo_local", symlink=False)
  ```
  (이미 실제 파일이 폴더 안에 있으므로 symlink=False로 복사해도 원본 재다운로드가 필요 없다.)
"""
with open(EXPORT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print(_json.dumps(manifest, ensure_ascii=False, indent=2))
print("README.md / manifest.json 저장 완료 →", EXPORT_DIR)


### 압축 (공유하기 쉽게 zip으로)

In [ ]:
import shutil

zip_path = shutil.make_archive(
    base_name=str(EXPORT_DIR),  # {EXPORT_DIR}.zip 생성
    format="zip",
    root_dir=EXPORT_DIR.parent,
    base_dir=EXPORT_DIR.name,
)
print("압축 완료:", zip_path)
print(
    "이 zip 파일을 팀원에게 공유하면 된다(Slack/Drive 등, 이 레포에 커밋하지 말 것 — .gitignore 확인)."
)


## 팀원 사용 방법 요약

1. 공유받은 zip 파일을 아무 위치에나 압축 해제한다(경로 무관).
2. 압축 해제한 폴더 안(또는 그 경로를 알고 있는 채로) 아래처럼 불러온다.

   ```python
   from src.data.subset import load_portable_coco

   train_coco = load_portable_coco("<압축해제경로>/train.json")
   val_coco = load_portable_coco("<압축해제경로>/val.json")
   ```

3. torchvision 계열 모델은 `train_coco`/`val_coco`를 그대로
   `src.train.train_torchvision(config, train_coco, val_coco, project_dir)`에
   넘기면 된다 — 이 프로젝트의 모든 학습 함수는 COCO dict를 입력으로 받도록
   통일돼 있어서 추가 변환이 필요 없다.
4. YOLO로 학습하려면 로컬에서 한 번만
   `prepare_yolo_dataset(train_coco, val_coco, "아무데나", symlink=False)`을
   실행해 YOLO 포맷(images/labels 폴더 + data.yaml)으로 변환하면 된다. 이미
   실제 이미지 파일이 폴더 안에 있으므로 `symlink=False`로 복사해도 원본을
   따로 받을 필요가 없다.
5. **주의**: 이 데이터는 `beamsearch/CJY`의 leakage-safe 파이프라인 결과다. 새
   AIHub 조합을 추가하는 등 원본 데이터를 바꾸고 싶으면 이 zip을 수정하지 말고
   `docs/data_leakage_prevention.md`의 파이프라인을 처음부터 다시 돌릴 것
   (그래야 새로 추가된 데이터도 동일하게 leakage 검증을 거친다).